<a href="https://colab.research.google.com/github/HariSharma10/Machine_Leaning/blob/main/Movie_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('tmdb_5000_movies.csv')
print(df.head())
print(df.info())
print(df.columns.tolist())

      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www.thedarkknightrises.com/   49026   
4          http://movies.disney.com/john-carter   49529   

                                            keywords original_language  \
0  [{"id": 1463, "name": "culture clash"}, {"id":...                en   
1  [{"id": 270, "name": "ocean"}, {"id": 726, "na...                en   
2  [{"id": 470, "nam

In [2]:
df = df[['title', 'overview', 'genres', 'keywords']]
df = df.dropna(subset=['overview'])
df['genres'] = df['genres'].fillna('')
df['keywords'] = df['keywords'].fillna('')
df.reset_index(drop=True, inplace=True)
print(df.head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   
3  Following the death of District Attorney Harve...   
4  John Carter is a war-weary, former military ca...   

                                              genres  \
0  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                           

In [3]:
import ast

def parse_json_column(text):
      try:
          items = ast.literal_eval(text)
          return ' '.join([item['name'] for item in items])
      except:
          return ''

df['genres'] = df['genres'].apply(parse_json_column)
df['keywords'] = df['keywords'].apply(parse_json_column)

# Combine everything into one text field for each movie
df['combined_features'] = df['overview'] + ' ' + df['genres'] + ' ' + df['keywords']
print(df['combined_features'].head())

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: combined_features, dtype: object


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Similarity matrix shape:", cosine_sim.shape)

TF-IDF matrix shape: (4800, 23003)
Similarity matrix shape: (4800, 4800)


In [6]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def recommend(title, num_recommendations=5):
    if title not in indices:
            return f"'{title}' not found in dataset."

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num_recommendations+1]  # skip itself

    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

In [7]:
print(recommend('The Dark Knight'))
print(recommend('Avatar'))

3       The Dark Knight Rises
428            Batman Returns
119             Batman Begins
1359                   Batman
299            Batman Forever
Name: title, dtype: object
373     Mission to Mars
2403             Aliens
1531          Moonraker
838              Alien³
2015         Spaceballs
Name: title, dtype: object
